In [ ]:
#!pip install missingno

In [ ]:
import os
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
from scipy import stats
import math


# 1. Bloquear advertencias a nivel de sistema antes de que pandas inicialice por completo
os.environ['PYTHONWARNINGS'] = 'ignore'

# 2. Forzar la supresión de advertencias de formato en Python y pandas
warnings.filterwarnings('ignore')
warnings.simplefilter(action='ignore', category=FutureWarning)

# Semilla global de reproducibilidad según las guías del curso
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Estándares de visualización
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12


# A1 - Análisis exploratorio de datos (EDA) para selección de variables y reducción de dimensionalidad

## Sección A: Identificación del conjunto de datos y justificación

El conjunto seleccionado para este análisis exploratorio de datos (EDA) es el **IEEE-CIS Fraud Detection Dataset**, proporcionado por Vesta Corporation (referente en soluciones de pago garantizado para comercio electrónico) y alojado por la IEEE Computational Intelligence Society.

En el comercio digital real, la detección de fraude es inherentemente compleja: desbalance extremo de clases, volumen masivo, alta dimensionalidad y gran cantidad de valores faltantes por privacidad o limitaciones del sistema. Este conjunto benchmark contiene transacciones de comercio electrónico a gran escala, con variables que van desde características del dispositivo hasta metadatos de la transacción, lo que lo hace ideal para estrategias rigurosas de selección de variables y reducción de dimensionalidad.

**Cita**

@misc{ieee-fraud-detection,
    author = {Addison Howard and Bernadette Bouchon-Meunier and IEEE CIS and inversion and John Lei and Lynn@Vesta and Marcus2010 and Prof. Hussein Abbass},
    title = {IEEE-CIS Fraud Detection},
    year = {2019},
    howpublished = {\url{https://kaggle.com/competitions/ieee-fraud-detection}},
    note = {Kaggle}
}

## Sección B: Diccionario de datos

El conjunto IEEE-CIS Fraud Detection sigue un esquema relacional con dos archivos principales enlazados por un identificador único de transacción: `train_transaction.csv` y `train_identity.csv`. Las variables se organizan, clasifican y definen operativamente a continuación según su tipo de dato y su escala estadística.

In [ ]:
# 1. Carga de archivos en DataFrames de Pandas
# Ajusta las rutas si tus archivos están en una subcarpeta, p. ej. 'data/'
train_transaction = pd.read_csv('train_transaction.csv')
train_identity = pd.read_csv('train_identity.csv')

print(f"Matriz de transacciones cargada: {train_transaction.shape[0]} filas, {train_transaction.shape[1]} variables.")
print(f"Matriz de identidad cargada: {train_identity.shape[0]} filas, {train_identity.shape[1]} variables.")

# 2. Combinación relacional mediante left join sobre la llave principal
# Garantiza no perder transacciones sin registros correspondientes en identidad
df_train = pd.merge(train_transaction, train_identity, on='TransactionID', how='left')

print("-" * 60)
print(f"Forma de la matriz EDA consolidada: {df_train.shape[0]} filas y {df_train.shape[1]} variables.")


In [ ]:
# 1. Lista de columnas representativas que cubren los bloques del esquema
representative_features = [
    'TransactionID', 'TransactionDT', 'TransactionAmt',
    'ProductCD', 'card1', 'card4', 'addr1', 'dist1',
    'P_emaildomain', 'M1', 'C1', 'D1', 'V1', 'V101',
    'DeviceType', 'DeviceInfo', 'id_01', 'id_13', 'id_19', 'id_38'
]

# 2. Filtrar a columnas presentes en el entorno de trabajo
active_features = [col for col in representative_features if col in df_train.columns]

# 3. Extraer metadatos estructurales desde el DataFrame
metadata_records = []
for col in active_features:
    storage_type = str(df_train[col].dtype)
    missing_count = df_train[col].isna().sum()

    # Tipo lógico según propiedades analíticas
    if storage_type in ['object', 'bool'] or col in ['TransactionID', 'isFraud', 'addr1', 'addr2']:
        logical_type = 'Categórica'
    else:
        logical_type = 'Numérica'

    metadata_records.append({
        'Nombre de variable': col,
        'Tipo lógico': logical_type,
        'Tipo almacenamiento (Python)': storage_type,
        'Nulos activos (conteo)': missing_count
    })

# 4. DataFrame de verificación tipológica
df_type_justification = pd.DataFrame(metadata_records)

# 5. Mostrar tabla
print("=== MATRIZ DE TIPOS DE ALMACENAMIENTO ===")
display(df_type_justification)

---

### B1 - Identificadores operativos clave y variable objetivo

| Nombre de variable | Tipo de dato (lógico) | Tipo de almacenamiento (Python) | Escala de medición | Descripción operativa y lógica de dominio |
| :--- | :--- | :--- | :--- | :--- |
| **TransactionID** | Categórica | Entero | Nominal (identificador) | Llave única por transacción. Sirve para unir las tablas de transacción e identidad. |
| **isFraud** | Categórica | Binaria / booleana | Nominal (objetivo) | Variable objetivo. Indica si la transacción fue fraudulenta (**1**) o legítima (**0**). |

---

### B2 - Variables de la tabla de transacciones (`train_transaction.csv`)

| Grupo de variables | Tipo de dato (lógico) | Tipo de almacenamiento (Python) | Escala de medición | Descripción operativa y lógica de dominio |
| :--- | :--- | :--- | :--- | :--- |
| **TransactionDT** | Numérica | Flotante / entero | Continua | Proxy temporal: segundos transcurridos desde una fecha-referencia no revelada (no es un sello temporal absoluto). |
| **TransactionAmt** | Numérica | Flotante | Continua | Monto del pago en USD (o equivalente en moneda local). |
| **ProductCD** | Categórica | Cadena / object | Nominal | Código o categoría de producto por transacción (p. ej. W, H, C, R). |
| **card1 - card6** | Categórica | Entero / cadena | Nominal | Información de tarjeta (tipo, red emisora, banco emisor aprox., país). |
| **addr1, addr2** | Categórica | Flotante / entero | Nominal | Códigos geográficos; `addr1` suele corresponder a región postal de facturación y `addr2` a país. |
| **P_emaildomain**<br>**R_emaildomain** | Categórica | Cadena / object | Nominal | Dominios de correo del comprador (**P**) y del receptor (**R**). |
| **M1 - M9** | Categórica | Cadena / object | Nominal | Indicadores de coincidencias (p. ej. T/F u otros códigos) entre datos de facturación, nombres y tarjetas. |
| **C1 - C14** | Numérica | Entero / flotante | Discreta | Conteos de comportamiento (cuántas veces se observa tarjeta, correo, dispositivo, etc.). |
| **D1 - D15** | Numérica | Flotante | Continua | Diferencias temporales (días o intervalos entre la transacción actual y registros previos). |
| **dist1, dist2** | Numérica | Flotante | Continua | Métricas de distancia física entre atributos (facturación, envío, códigos postales, IP, teléfono, según ingeniería de la tabla). |

---

### B3 - Variables comportamentales V de Vesta (bloque V)

| Grupo de variables | Tipo de dato (lógico) | Tipo de almacenamiento (Python) | Escala de medición | Descripción operativa y lógica de dominio |
| :--- | :--- | :--- | :--- | :--- |
| **V1 - V339** | Numérica | Flotante | Continua / discreta | Variables anonimizadas e ingenierizadas por Vesta: clasificaciones, conteos y puntuaciones de relación entre atributos de pago y comportamiento histórico. |

---

### B4 - Variables de la tabla de identidad (`train_identity.csv`)

| Grupo de variables | Tipo de dato (lógico) | Tipo de almacenamiento (Python) | Escala de medición | Descripción operativa y lógica de dominio |
| :--- | :--- | :--- | :--- | :--- |
| **DeviceType** | Categórica | Cadena / object | Nominal | Entorno del cliente (móvil, escritorio). |
| **DeviceInfo** | Categórica | Cadena / object | Nominal | Cadena de hardware/software (p. ej. Windows, iOS, modelo de dispositivo). |
| **id_01 - id_11** | Numérica | Flotante | Continua / discreta | Marcadores numéricos (red, resolución, señales de comportamiento técnico). |
| **id_12 - id_38** | Categórica | Cadena / object | Nominal | Indicadores categóricos de identidad digital (navegador, proxy, SO, disponibilidad de credenciales, etc.). |

## Sección C: Evaluación de la calidad de datos

### C1 - Evaluación de registros duplicados

Antes del perfil descriptivo o pruebas de ausencia sistemática conviene revisar la integridad básica del conjunto. Este paso cuantifica duplicidad de filas en el conjunto fusionado usando la llave principal (**TransactionID**). Detectar y eliminar duplicados evita distorsionar la varianza y los análisis univariados posteriores.

In [ ]:
# Sección C: Evaluación de la calidad de datos
# Subsección: Registros duplicados

# 1. Contar duplicados exactos según TransactionID
duplicate_count = df_train.duplicated(subset=['TransactionID']).sum()
duplicate_percentage = (duplicate_count / len(df_train)) * 100

print("--- Evaluación de registros duplicados ---")
print(f"Duplicados detectados en total: {duplicate_count}")
print(f"Porcentaje de duplicados estructurales: {duplicate_percentage:.4f}%")

# 2. Limpieza programática si aplica
if duplicate_count > 0:
    df_train = df_train.drop_duplicates(subset=['TransactionID'])
    print("Se eliminaron filas duplicadas para preservar una fila única por transacción.")
else:
    print("No se requirió limpieza por duplicados. Cada fila representa un evento único.")


### C2 - Codificación inconsistente y canonicalización de cadenas

Las entradas de texto inconsistentes (espacios finales o mayúsculas/minúsculas mezcladas) pueden inflar artificialmente la cardinalidad de campos categóricos. Se revisan variables clave (`ProductCD`, `card4`, `DeviceType`) para confirmar etiquetas limpias y estables.

In [ ]:
# Subsección: Evaluación de codificación inconsistente

# Variables categóricas para inspección
categorical_integrity_checks = ['ProductCD', 'card4', 'DeviceType']

print("--- Perfil de consistencia categórica ---")
for feature in categorical_integrity_checks:
    if feature in df_train.columns:
        unique_labels = df_train[feature].unique()
        raw_cardinality = df_train[feature].nunique()

        print(f"\nPerfil de '{feature}' | cardinalidad: {raw_cardinality}")
        print(f"Etiquetas únicas: {unique_labels}")

        if df_train[feature].dtype == 'object':
            normalized_count = df_train[feature].astype(str).str.strip().str.lower().nunique()
            if normalized_count != raw_cardinality:
                print(f"⚠️ Posible inconsistencia espacial/formato en '{feature}'.")


### C3 - Valores fuera de rango y diagnóstico de valores atípicos

Se aísla el indicador financiero (**TransactionAmt**) para validar el dominio (p. ej. montos no negativos y sentido económico) y fijar un umbral de valores extremos mediante el método del rango intercuartílico (RIC / IQR).

In [ ]:
# Sección C: Evaluación de la calidad de datos
# Subsección: Diagnóstico visual de atípicos con diagramas de caja

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Gráfico A: Monto de transacción (escala cruda) ---
sns.boxplot(data=df_train, x='TransactionAmt', ax=axes[0], color='teal')
axes[0].set_title("Distribución y valores atípicos del monto de transacción (escala original)")
axes[0].set_xlabel("Monto de transacción (USD)")

# --- Gráfico B: Escala logarítmica ---
sns.boxplot(data=df_train, x='TransactionAmt', ax=axes[1], color='lightblue')
axes[1].set_xscale('log')
axes[1].set_title("Perfil de atípicos del monto (escala log)")
axes[1].set_xlabel("Log del monto de transacción (USD)")

plt.tight_layout()
plt.show()

# Límites matemáticos IQR
Q1 = df_train['TransactionAmt'].quantile(0.25)
Q3 = df_train['TransactionAmt'].quantile(0.75)
IQR = Q3 - Q1
upper_bound = Q3 + 1.5 * IQR

extreme_outliers_count = df_train[df_train['TransactionAmt'] > upper_bound].shape[0]

print("--- Límites matemáticos por rango intercuartílico (IQR) ---")
print(f" - Cuartil inferior (Q1): {Q1:.2f} USD")
print(f" - Cuartil superior (Q3): {Q3:.2f} USD")
print(f" - IQR: {IQR:.2f} USD")
print(f" - Límite superior (Q3 + 1.5×IQR): {upper_bound:.2f} USD")
print(f" - Observaciones por encima del límite: {extreme_outliers_count}")



Basándome en los diagnósticos visuales y cuantitativos, las 66 482 observaciones identificadas como valores atípicos extremos **no** se eliminan del conjunto por las siguientes razones:

* **Preservación de señal de fraude:** En pagos electrónicos, valores altos pueden asociarse a fraude; los agentes pueden intentar movimientos de gran monto antes de que la tarjeta sea bloqueada. Eliminar esos casos borraría señales que los modelos deben aprender.
* **Datos realistas:** No se trata de errores de captura necesariamente sino de eventos creíbles de alto valor.
* **Mitigación aguas abajo:** En lugar de depuración por borrado, la asimetría extrema se abordará con transformaciones robustas (`log`, `RobustScaler`) y modelos poco sensibles a atípicos (bosques aleatorios, LightGBM, etc.).

### C4 - Contraste de mecanismos de ausencia de datos (MCAR frente a MAR/MNAR)

#### DeviceType

Evalúo qué patrón estadístico rige los datos ausentes: no todas las transacciones generan bitácoras de identidad. Se plantea la hipótesis de que los faltantes (p. ej. en **DeviceType**) dependen estructuralmente de la clase **isFraud** (MAR o MNAR) y no de un proceso puramente aleatorio (MCAR). Para la rúbrica se ejecuta chi-cuadrado de independencia reportando estadístico y valor p.

In [ ]:
# Subsección: Prueba de patrones de ausencia
# H0: la ausencia en identidad es independiente de la clase objetivo (MCAR).
# H1: la ausencia depende de la legitimidad de la transacción (MAR o MNAR).

print("--- Inferencia estadística sobre ausencia de datos ---")

df_train['DeviceType_Missing'] = df_train['DeviceType'].isna()

contingency_matrix = pd.crosstab(df_train['DeviceType_Missing'], df_train['isFraud'])
print("\nTabla de contingencia (DeviceType ausente vs. isFraud):")
print(contingency_matrix)

chi2_stat, p_value, dof, expected_freq = stats.chi2_contingency(contingency_matrix)

print("\n--- Resultados chi-cuadrado ---")
print(f"Estadístico chi-cuadrado: {chi2_stat:.4f}")
print(f"Valor p asintótico: {p_value:.4e}")


**Conclusión**: Se rechaza H0 al 95 % de confianza. Los faltantes de `DeviceType` dependen de `isFraud`, lo cual refuta MCAR y apunta a un mecanismo **MAR/MNAR** (p. ej. captura diferencial de especificaciones o rutas donde no se exige huella de identidad).

#### Proximidad espacial (dist1)

Para contrastar patrones entre bloques independientes replique el mismo enfoque con **dist1** (métrica de distancia física declarada por la tabla). Suele aparecer ausencia por límites de enruteo o fronteras transfronterizas. La hipótesis formal es que los faltantes en `dist1` se asocian estadísticamente con `isFraud`, estableciendo contexto MAR distinto al canal de identidad.

In [ ]:
# 1. Indicador ausente-binario para dist1
df_train['dist1_Missing'] = df_train['dist1'].isna()

# 2. Tabla vs. clase objetivo
dist_contingency = pd.crosstab(df_train['dist1_Missing'], df_train['isFraud'])
print("\nTabla de contingencia (dist1 ausente vs. isFraud):")
print(dist_contingency)

# 3. Chi-cuadrado
chi2_stat_dist, p_value_dist, dof_dist, expected_dist = stats.chi2_contingency(dist_contingency)

print("\n--- Resultados chi-cuadrado (dist1) ---")
print(f"Estadístico chi-cuadrado: {chi2_stat_dist:.4f}")
print(f"Valor p asintótico: {p_value_dist:.4e}")


**Conclusión**: Se rechaza la hipótesis nula (H0) al 95 % de confianza. El patrón de ausencia en `dist1` depende de `isFraud`; el proceso no cumple MCAR. Es compatible con **MAR**, lo que sugiere que algunos flujos de pago/productos pueden no calcular distancia física y, a la vez, presentar probabilidades diferenciales de fraude.

In [ ]:
# 1. Indicadores de ausencia
df_train['DeviceType_Missing'] = df_train['DeviceType'].isna()
df_train['dist1_Missing'] = df_train['dist1'].isna()

# 2. Tasas de ausencia por clase de fraude
dev_missing_rates = df_train.groupby('isFraud')['DeviceType_Missing'].mean().reset_index()
dist_missing_rates = df_train.groupby('isFraud')['dist1_Missing'].mean().reset_index()

# 3. Gráficas de barras comparativas
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Gráfico A: DeviceType
sns.barplot(
    data=dev_missing_rates,
    x='isFraud',
    y='DeviceType_Missing',
    ax=axes[0],
    palette='coolwarm',
    hue='isFraud',
    legend=False,
)
axes[0].set_title(
    "Probabilidad empírica de ausencia — matrices de identidad (DeviceType)"
)
axes[0].set_xlabel("Clase (0: legítimo, 1: fraude)")
axes[0].set_ylabel("Proporción de filas con valor nulo")
axes[0].set_ylim(0, 1.0)

for p in axes[0].patches:
    axes[0].annotate(
        f'{100 * p.get_height():.2f}%',
        (p.get_x() + p.get_width() / 2.0, p.get_height() + 0.02),
        ha='center',
        va='center',
        fontsize=11,
        color='black',
        fontweight='bold',
    )

# Gráfico B: dist1
sns.barplot(
    data=dist_missing_rates,
    x='isFraud',
    y='dist1_Missing',
    ax=axes[1],
    palette='coolwarm',
    hue='isFraud',
    legend=False,
)
axes[1].set_title("Probabilidad empírica de ausencia — distancia espacial (dist1)")
axes[1].set_xlabel("Clase (0: legítimo, 1: fraude)")
axes[1].set_ylabel("Proporción de filas con valor nulo")
axes[1].set_ylim(0, 1.0)

for p in axes[1].patches:
    axes[1].annotate(
        f'{100 * p.get_height():.2f}%',
        (p.get_x() + p.get_width() / 2.0, p.get_height() + 0.02),
        ha='center',
        va='center',
        fontsize=11,
        color='black',
        fontweight='bold',
    )

plt.tight_layout()
plt.show()


La visualización bifocal refuta el supuesto MCAR (**Missing Completely at Random**) y muestra ausencias muy informativas:

* **`DeviceType` (identidad digital):** en operaciones presumiblemente legítimas la tasa de nulos llega en torno al **77,22 %**, mientras entre fraudulentas cae a **45,74 %**. El patrón sugiere **MAR/MNAR**, asociado a rutas donde el motor de seguridad sólo registra huella cuando el score de riesgo es alto.

* **`dist1` (proximidad espacial):** la ausencia sube desde **47,28 %** (no fraude) a **65,17 %** en fraude, compatible con rutas donde redirecciones tipo VPN o infraestructura transfronteriza impiden obtener distancias.

Al ser predictivas estas ausencias conviene preservar banderas de faltantes o tratamientos que respeten la dispersión dentro de clasificadores en árbol.

## Sección D: Análisis exploratorio de datos

### D1 — Análisis univariado: variables numéricas

#### TransactionAmt y TransactionDT

Analizo el comportamiento marginal de **`TransactionAmt`** (monto) y **`TransactionDT`** (eje temporal relativo): histogramas con estimación KDE y diagramas de caja para sintetizar asimetría y colas pesadas.

In [ ]:
# Sección D: exploración — D1 variables numéricas

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# TransactionAmt — histogramas
sns.histplot(
    data=df_train, x='TransactionAmt', bins=100, kde=True, ax=axes[0, 0], color='teal'
)
axes[0, 0].set_title("Distribución del monto (escala original)")
axes[0, 0].set_xlabel("Monto de transacción (USD)")
axes[0, 0].set_ylabel("Frecuencia")

sns.histplot(
    data=df_train, x='TransactionAmt', bins=100, kde=True, ax=axes[0, 1], color='teal'
)
axes[0, 1].set_yscale('log')
axes[0, 1].set_title("Distribución del monto — escala log en frecuencia")
axes[0, 1].set_xlabel("Monto de transacción (USD)")
axes[0, 1].set_ylabel("Log(frecuencia)")

# TransactionDT
sns.histplot(
    data=df_train, x='TransactionDT', bins=50, kde=True, ax=axes[1, 0], color='darkblue'
)
axes[1, 0].set_title("Distribución del delta temporal TransactionDT")
axes[1, 0].set_xlabel("Tiempo transcurrido (segundos desde epoch de referencia)")
axes[1, 0].set_ylabel("Frecuencia")

sns.boxplot(data=df_train, x='TransactionDT', ax=axes[1, 1], color='lightblue')
axes[1, 1].set_title("Diagrama de cuartiles — TransactionDT")
axes[1, 1].set_xlabel("Tiempo transcurrido (segundos)")

plt.tight_layout()
plt.show()

print("--- Estadísticas descriptivas ---")
print(df_train[['TransactionAmt', 'TransactionDT']].describe())


#### dist1, C1 y C2

Construyo histogramas con KDE y diagramas de caja para `dist1` (proximidad), `C1` y `C2` (conteos de comportamiento). La fuerte asimetría fuerza vistas lineales antes de aplicar ejes transformados donde el lienzo saturado lo requiera.

In [ ]:
# Sección D — D1 (vistas en escala lineal cruda)


fig, axes = plt.subplots(3, 2, figsize=(16, 15))

# dist1
sns.histplot(data=df_train, x='dist1', bins=50, kde=True, ax=axes[0, 0], color='teal')
axes[0, 0].set_title("Distribución de dist1 (escala original)")
axes[0, 0].set_xlabel("Métrica de distancia")

sns.boxplot(data=df_train, x='dist1', ax=axes[0, 1], color='lightblue')
axes[0, 1].set_title("Diagrama de caja — dist1 (linear)")
axes[0, 1].set_xlabel("Distancia")

# C1
sns.histplot(data=df_train, x='C1', bins=50, kde=True, ax=axes[1, 0], color='darkblue')
axes[1, 0].set_title("Distribución del conteo C1 (escala original)")
axes[1, 0].set_xlabel("Frecuencias / conteos")

sns.boxplot(data=df_train, x='C1', ax=axes[1, 1], color='royalblue')
axes[1, 1].set_title("Diagrama de caja — C1 (linear)")
axes[1, 1].set_xlabel("Valor de C1")

# C2
sns.histplot(data=df_train, x='C2', bins=50, kde=True, ax=axes[2, 0], color='purple')
axes[2, 0].set_title("Distribución del conteo C2 (escala original)")
axes[2, 0].set_xlabel("Frecuencias / conteos")

sns.boxplot(data=df_train, x='C2', ax=axes[2, 1], color='plum')
axes[2, 1].set_title("Diagrama de caja — C2 (linear)")
axes[2, 1].set_xlabel("Valor de C2")

plt.tight_layout()
plt.show()


In [ ]:
# Sección D — D1 con transformaciones para interpretación

fig, axes = plt.subplots(3, 2, figsize=(16, 15))

# dist1
sns.histplot(data=df_train, x='dist1', bins=50, kde=True, ax=axes[0, 0], color='teal')
axes[0, 0].set_yscale('log')
axes[0, 0].set_title("Distribución de dist1 — escala log en frecuencia")
axes[0, 0].set_xlabel("Métrica de distancia")

sns.boxplot(data=df_train, x='dist1', ax=axes[0, 1], color='lightblue')
axes[0, 1].set_xscale('log')
axes[0, 1].set_title("Cuartiles — dist1 (log en variable)")
axes[0, 1].set_xlabel("Distancia (log)")

# C1
sns.histplot(data=df_train, x='C1', bins=50, kde=True, ax=axes[1, 0], color='darkblue')
axes[1, 0].set_yscale('log')
axes[1, 0].set_title("Distribución de C1 — log en conteo de barras")
axes[1, 0].set_xlabel("Valores numéricos de C1")

sns.boxplot(data=df_train, x='C1', ax=axes[1, 1], color='royalblue')
axes[1, 1].set_xscale('log')
axes[1, 1].set_title("Diagrama de caja — C1")
axes[1, 1].set_xlabel("Valor de C1 (escala log)")

# C2
sns.histplot(data=df_train, x='C2', bins=50, kde=True, ax=axes[2, 0], color='purple')
axes[2, 0].set_yscale('log')
axes[2, 0].set_title("Distribución de C2 — escala log en frecuencia")
axes[2, 0].set_xlabel("Valores numéricos de C2")

sns.boxplot(data=df_train, x='C2', ax=axes[2, 1], color='plum')
axes[2, 1].set_xscale('log')
axes[2, 1].set_title("Diagrama de caja — C2")
axes[2, 1].set_xlabel("Valor de C2 (escala log)")

plt.tight_layout()
plt.show()

display(df_train[['dist1', 'C1', 'C2']].describe())


Durante el seguimiento univariado (`dist1`, `C1`, `C2`) la vista lineal provocó el típico *efecto rascacielos*: más del 95 % de las observaciones colapsan en los primeros contenedores ante colas muy pesadas. Para cumplir estándares de EDA aplico **ejes transformados logarítmicos** donde aportó claridad (frecuencia o valores según caso).

### D2 — Análisis univariado: variables categóricas

Selecciono un conjunto nominal representativo cubriendo transacciones (`ProductCD`), tarjetas (`card4`, `card6`), verificación (`M4`) y marca dispositivo/identidad (`DeviceType`, `id_12`, `id_15`, `id_38`).

In [ ]:
from IPython.display import display, HTML

disable_scroll_css = '''
<style>
    .output_scroll, .output_wrapper, .jp-OutputArea-child, .jp-Cell-outputArea {
        height: auto !important;
        max-height: none !important;
        overflow: visible !important;
    }
</style>
'''
display(HTML(disable_scroll_css))

categorical_pool = [
    'ProductCD', 'card4', 'card6', 'M4',
    'DeviceType', 'id_12', 'id_15', 'id_38'
]

active_categorical = [col for col in categorical_pool if col in df_train.columns]
num_plots = len(active_categorical)
num_cols = 2
num_rows = math.ceil(num_plots / num_cols)

fig, axes = plt.subplots(num_rows, num_cols, figsize=(16, 6 * num_rows))
axes = axes.flatten()

total_observations = len(df_train)

for i, col in enumerate(active_categorical):
    ax = axes[i]

    sorted_order = df_train[col].dropna().value_counts().index

    sns.countplot(
        data=df_train,
        x=col,
        hue=col,
        order=sorted_order,
        ax=ax,
        palette='viridis' if i % 2 == 0 else 'magma',
        legend=False,
    )

    ax.set_title(
        f"Distribución empírica: {col}", fontsize=13, fontweight='bold'
    )
    ax.set_xlabel(f"Categorías de {col}", fontsize=11)
    ax.set_ylabel("Observaciones", fontsize=11)

    try:
        max_bar_height = df_train[col].value_counts().max()
        y_offset = max_bar_height * 0.03
    except Exception:
        y_offset = 1000

    for p in ax.patches:
        h = p.get_height()
        if h > 0:
            percentage_string = f'{100 * h / total_observations:.2f}%'
            ax.annotate(
                percentage_string,
                (p.get_x() + p.get_width() / 2.0, h + y_offset),
                ha='center',
                va='bottom',
                fontsize=10,
                color='black',
                fontweight='bold',
            )

    if len(sorted_order) > 3:
        ax.tick_params(axis='x', rotation=20)

for j in range(num_plots, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

print("=== MÉTRICAS DE DISTRIBUCIONES CATEGÓRICAS ===")
for col in active_categorical:
    print(f"\nFrecuencias absolutas '{col}':")
    print(df_train[col].value_counts(dropna=False))


### D3 — Relaciones bivariadas numérica–numérica

Analizo la co-distribución marginal de métricas clave usando **Spearman ρ** frente a Pearson: así captamos relaciones monótonas bajo asimetría severa/outliers (`TransactionAmt`, `TransactionDT`, `C1`/`C2` y un ejemplo `V101`–`V103`). Complemento la matriz con dispersión objetivo montos vs tiempo en escala log para el eje monetario.

In [ ]:
# Sección D — D3 correlaciones numericas

numerical_subset = [
    'TransactionAmt', 'TransactionDT',
    'C1', 'C2',
    'V101', 'V102', 'V103'
]
valid_numerical_features = [col for col in numerical_subset if col in df_train.columns]

correlation_matrix = df_train[valid_numerical_features].corr(method='spearman')

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

sns.heatmap(
    data=correlation_matrix,
    annot=True,
    fmt=".3f",
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    square=True,
    ax=axes[0],
    cbar_kws={"label": "Coeficiente de correlacion rho (Spearman)"},
)
axes[0].set_title("Heatmap rho de Spearman (subconjunto numerico)")

sns.scatterplot(
    data=df_train,
    x='TransactionDT',
    y='TransactionAmt',
    alpha=0.1,
    ax=axes[1],
    color='darkblue',
)
axes[1].set_yscale('log')
axes[1].set_title("Dispersion: monto vs delta temporal TransactionDT")
axes[1].set_xlabel("Segundos transcurridos (epoch referencia)")
axes[1].set_ylabel("Logaritmo decimal del monto (USD)")

plt.tight_layout()
plt.show()

print("--- Matriz rho de Spearman ---")
print(correlation_matrix)


**Conclusiones**
========================>

Las correlaciones de Spearman indican alta asociacion monotona entre `V101`/`V102`/`V103` y fuerte relacion positiva tambien entre `C1` y `C2`. En cambio, `TransactionAmt` y `TransactionDT` apenas acumulan señales monotonicas coherentes contra el cuadro mostrado, lo cual concuerda con el diagrama disperso donde los montos muestran ciclos de densidad pero no tendencia marcada contra el tiempo relativo.

### D4 — Relaciones bivariadas variable ↔ objetivo `isFraud`

Se estratifican variables continuas frente al objetivo binario y se grafican cruces categoricos cuando aplica (`ProductCD`). Cada grafico viene acompanado formalmente por pruebas Mann-Whitney U y chi cuadrado declarando estadistico y valor p.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

sns.boxplot(
    data=df_train,
    x='isFraud',
    y='TransactionAmt',
    ax=axes[0],
    palette='Set2',
    hue='isFraud',
    legend=False,
)
axes[0].set_yscale('log')
axes[0].set_title("Distribucion del monto segun clase de fraude")
axes[0].set_xlabel("Clase (0 legítimo, 1 fraude)")
axes[0].set_ylabel("Logaritmo decimal del monto (USD)")

product_fraud_rates = (
    df_train.groupby('ProductCD')['isFraud'].mean().reset_index().sort_values(by='isFraud', ascending=False)
)

sns.barplot(
    data=product_fraud_rates,
    x='ProductCD',
    y='isFraud',
    ax=axes[1],
    palette='viridis',
    hue='ProductCD',
    legend=False,
)
axes[1].set_title("Tasa observada de fraude por ProductCD")
axes[1].set_xlabel("Codigo ProductCD")
axes[1].set_ylabel("Fracción fraudulenta")

for p in axes[1].patches:
    pct = f'{100 * p.get_height():.2f}%'
    axes[1].annotate(
        pct,
        (
            p.get_x() + p.get_width() / 2.0,
            p.get_height() + 0.002,
        ),
        ha='center',
        va='center',
        fontsize=11,
        color='black',
        fontweight='bold',
    )

plt.tight_layout()
plt.show()

print("=" * 70)
print("--- PRUEBAS DE HIPOTESIS ---")
print("=" * 70)

amt_ok = df_train[df_train['isFraud'] == 0]['TransactionAmt']
amt_fraud = df_train[df_train['isFraud'] == 1]['TransactionAmt']
mwu_stat, mwu_p = stats.mannwhitneyu(amt_ok, amt_fraud, alternative='two-sided')
print("\n[Test 1] Mann-Whitney U (Montos vs isFraud):")
print(f" - Estadistico U: {mwu_stat:.4f}")
print(f" - p-valor asintótico: {mwu_p:.4e}")
if mwu_p < 0.05:
    print(" - Decision: rechazar H0 — los montos difieren entre categorias observadas.")
else:
    print(" - Decision: no rechazar H0 — sin evidencia de cambio marcado.")

product_contingency = pd.crosstab(df_train['ProductCD'], df_train['isFraud'])
chi2_stat, chi2_p, dof, expected = stats.chi2_contingency(product_contingency)

print("\n[Test 2] Chi cuadrado ProductCD × isFraud:")
print(f" - Chi cuadrado observado: {chi2_stat:.4f}")
print(f" - p-valor: {chi2_p:.4e}")
print(f" - Grados de libertad: {dof}")
if chi2_p < 0.05:
    print(" - Decision: rechazar H0 — las tasas de fraude cambian sistematicamente con producto.")
else:
    print(" - Decision: no rechazar H0 — tasas estadisticamente estables contra producto nominal.")


**Conclusiones**

==============>

Los paneles sugieren efectos diferentes por canal financiero cuando el problema se expresa mediante tasas y la significancia global proviene de chi cuadrado con multitud de grados libertad por categorias dispersas — conviene cualificar tamaño efecto usando métricas de negocio o modelos siguientes antes de extrapolar causa raiz especifica.

## Sección E: Formulación del problema




Basándome en el EDA y, para el marco conceptual, prácticas de estratificación con etiquetas retrospectivas similares a las discutidas en literatura prognóstica (p. ej. Laqueur et al., 2022), definimos formalmente el problema de datos.


---

### 1. Contexto operativo y de negocio
En pagos electrónicos el fraude implica pérdidas de ingreso, cargas antifraude y deterioro de la confianza de clientes e instituciones (incluyendo procesadores como Vesta en el mismo ecosistema de la competencia). Por el contrario, reglas sobre-agresivas elevan falsos rechazos y fricción al declinar cargos válidos.


El objetivo operativo principal es usar, a gran escala, registros administrativos de transacción e identificación digital anonimizada para aislar sistemáticamente señales de fraude respecto comportamientos de compra benignos típicamente legítimos.


---

### 2. Formalización estadística / machine learning

Se formula como una **clasificación supervisada binaria** bajo **desbalance marcado de clases** (aprox. 3.5 % de positivos).


* **Instancia observada \(i\)**: consolidación mediante la llave única después de hacer merge transacciones con identidades.

* **Espacio predictor \(X\)**: vector \( X_i\in\mathbb{R}^{d}\) con \( d\approx433 \) combinando valores financieros (`TransactionAmt`), tiempos relativos (`TransactionDT`, grupos `D`), conteos comportamentales (`C`), chequeos texto (`M`) y marcadores tecnológicos (`DeviceType`, `DeviceInfo`).



* **Variable objetivo \(Y_i\)**: etiqueta binaria igual que define el archivo de competencia:

  $$Y_i = \begin{cases} 
  1 & \text{si la historia del dataset marca fraude;} \\ 
  0 & \text{si la marca es transacción legítima.}
  \end{cases}$$

Propósito: estimar probabilidades \(\hat{p}_i = \mathbb{P}(Y_i=1 \mid X_i)\) mediante modelos adecuados a alta dimensionalidad (por ejemplo ensembles en árbol o modelos lineales regularizados, eventualmente después de PCA u otra reducción controlada).


---

### 3. Ventana temporal (lectura habitual del problema)

* **Momento del score:** \(\hat{p}_i\) debe interpretarse con la información observable en el mismo instante relativo sintetizado por `TransactionDT`.
* **Formación tardía del objetivo \(Y\):** positivos habitualmente llegan mediante confirmaciones post-transacción (p. ej. chargebacks registrados después), preservando orden causal entre rasgos conocidos primeramente y adjudicaciones posteriores.


---

### 4. Métricas de rendimiento conscientes del desbalance

Accuracy simple es engañosa cuando la clase mayoritaria domina conteos muestrales (modelos triviales predicen sólo clase negativa y logran alta exactitud).


1. **AUC ROC (AUROC)** para ordenar probabilidades discriminando globalmente.
2. **AUC PR (AUPRC)** como métrica complementaria enfocada a la clase minoritaria ante prevalencia baja.
3. **F-score y herramientas tipo índice de Youden** para fijar umbrales operativos condicionados a políticas corporativas (maximizar detección frente a contener rechazos equivocados).


